In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("modelling_dataset.parquet", engine="fastparquet").sample(500000, random_state=42)
df.head(20)


In [ ]:
df["accident_binary"] = (df["Accident_Count"] > 0).astype(int)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

split_80 = df["Time_Stamp"].quantile(0.80)

train_df = df[df["Time_Stamp"] < split_80].copy()
test_df  = df[df["Time_Stamp"] >= split_80].copy()

In [ ]:
baseline = train_df.groupby(["Zone_ID", "Hour"])["accident_binary"].mean()

In [ ]:

test_df = test_df.merge(
    baseline.rename("baseline_prob"),
    on=["Zone_ID", "Hour"],
    how="left"
)

In [ ]:
global_mean = train_df["accident_binary"].mean()
test_df["baseline_prob"] = test_df["baseline_prob"].fillna(global_mean)

In [ ]:
roc_auc = roc_auc_score(test_df["accident_binary"], test_df["baseline_prob"])
avg_p = average_precision_score(test_df["accident_binary"],test_df["baseline_prob"])
print("Baseline ROC-AUC:", roc_auc)
print("AP:", avg_p)

In [ ]:
threshold = 0.2
test_df["baseline_pred"] = (test_df["baseline_prob"] >= threshold).astype(int)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = test_df["accident_binary"]
y_pred = test_df["baseline_pred"]

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))